# LifeMate 2026: GEPA Mapper của Kiệt (prompt Mapper trên Qwen3-4B)

**Mục tiêu:** tối ưu prompt Mapper (sửa lỗi ASR và phương ngữ) bằng GEPA với metric nhanh (không chạy Planner), rồi chấm ablation 3 cấu hình bằng điểm cuối với Planner tốt nhất của Vương (`planner_v4`, 0.3597).

### Yêu cầu trên Kaggle (giống `run_dev_baseline.ipynb`):
1. **Settings** -> **Accelerator**: **GPU T4 x2** hoặc **GPU P100**.
2. **Settings** -> **Internet**: **ON**.
3. **Data**: Dataset chứa `train` và `public_test` của BTC, **có file audio**.

Ước tính GPU: ASR (đo khi chạy) + GEPA Mapper proxy (~1 giờ) + ablation 5 cấu hình x 100 mẫu (~3,5 giờ). Kaggle ngắt thì chạy lại cell đó: ASR và GEPA tự chạy tiếp từ `runs/`.

In [ ]:
# 1. Cài thư viện
!pip install -q -U vllm gepa pyyaml chunkformer

In [ ]:
# 2. Clone repo và chuyển sang nhánh gepa/kiet
import os
if not os.path.exists("ZAIC_2026"):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git
%cd ZAIC_2026
!git checkout gepa/kiet
!git pull origin gepa/kiet

In [ ]:
# 3. Thiết lập đường dẫn dữ liệu data/
import os, shutil
from pathlib import Path

os.makedirs("data", exist_ok=True)

# Tự động tìm thư mục data trong /kaggle/input
kaggle_input = Path("/kaggle/input")
found_train = list(kaggle_input.rglob("train.jsonl"))
if found_train:
    src_train_dir = found_train[0].parent
    src_root = src_train_dir.parent
    print(f"Tìm thấy dữ liệu tại: {src_root}")
    if not os.path.exists("data/train"):
        os.symlink(str(src_train_dir), "data/train")
    pub_dirs = list(kaggle_input.rglob("public_test.jsonl"))
    if pub_dirs and not os.path.exists("data/public_test"):
        os.symlink(str(pub_dirs[0].parent), "data/public_test")
    print("Đã liên kết dữ liệu vào data/:")
    !ls data
else:
    print("LƯU Ý: Vui lòng upload dataset train+public_test vào Kaggle và kiểm tra lại đường dẫn!")

In [ ]:
# 4. Unit test (không cần GPU)
!python tests/test_gepa_metric.py

In [ ]:
# 5. ASR một lần cho mọi lượt user (ChunkFormer của pipeline) -> runs/asr/train_asr.jsonl
os.makedirs("runs", exist_ok=True)
!python -m optimize.gepa.kiet.run_gepa asr --root data

In [ ]:
# 6. Kiểm tra nhanh ASR so với audio_text trên dev (không cần GPU): Mapper chỉ có ích nếu ASR có lỗi
import json, sys
sys.path.insert(0, ".")
from optimize.gepa.metric.metric import wer
from optimize.gepa.kiet.run_gepa import load_asr
asr = load_asr("runs/asr/train_asr.jsonl")
dev = set(open("eval/splits/dev_ids.txt").read().split())
items = [json.loads(l) for l in open("data/train/train.jsonl", encoding="utf-8")]
w = [wer(asr[i["id"]][-1][0], i["audio_text"]) for i in items if i["id"] in dev and i["id"] in asr]
print(f"WER trung bình {sum(w)/len(w):.3f}, mẫu có lỗi {sum(x > 0 for x in w)}/{len(w)}")

In [ ]:
# 7. GEPA cho Mapper (mapper_metric: proxy trong config: chỉ chạy Mapper, không chạy Planner nên rẻ GPU)
!python -m optimize.gepa.kiet.run_gepa run --root data --component mapper

In [ ]:
# 8. Ablation bằng điểm cuối trên 100 mẫu dev đầu (cùng tập cho mọi cấu hình), Planner = planner_v4 của Vương
import re
from pathlib import Path
vs = sorted(Path("prompts/gepa/kiet").glob("mapper_v*.txt"), key=lambda p: int(re.findall(r"\d+", p.stem)[0]))
best = vs[-1]
print("mapper tốt nhất:", best)
!python -m optimize.gepa.kiet.run_gepa eval --root data --prompt prompts/gepa/vuong/planner_v4.txt --limit 100
!python -m optimize.gepa.kiet.run_gepa eval --root data --asr --limit 100
!python -m optimize.gepa.kiet.run_gepa eval --root data --mapper old --limit 100
!python -m optimize.gepa.kiet.run_gepa eval --root data --mapper prompts/gepa/kiet/mapper_v0.txt --limit 100
if best.name != "mapper_v0.txt":
    !python -m optimize.gepa.kiet.run_gepa eval --root data --mapper {best} --limit 100
!cat prompts/gepa/kiet/manifest.json

In [ ]:
# 9. Đóng gói kết quả (prompt + manifest để commit; runs/ có raw và texts từng lượt cho Khánh test entity guard)
!zip -qr /kaggle/working/gepa_kiet_mapper.zip prompts/gepa/kiet runs/gepa_kiet runs/asr
print("Hoàn tất! Tải gepa_kiet_mapper.zip ở mục Output của Kaggle.")